**This Notebook is to check if the given dataset is in the pattern or not**

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "..." # <--- put file path here for example C:\DatasetTest\data1.csv or just file name if the dataset file is in the same folder as this notebook for example data1.csv

REQUIRED_COLUMNS = [
    "occurred",
    "mc_status",
    "mc_no",
    "process",
]

OPTIONAL_COLUMNS = [
    "registered",
]

# ============================================================
# EXPECTED DATASET EXAMPLE
# ============================================================

expected_format_example = pd.DataFrame({
    "occurred": [
        "2026-05-05 10:12:44",
        "2026-05-05 10:12:48",
        "2026-05-05 10:12:50",
        "2026-05-05 10:12:56",
        "2026-05-05 10:12:56",
    ],
    "mc_status": [
        "mc_fullWork",
        "mc_noWork",
        "mc_fullWork",
        "mc_explode",
        "mc_alarm",
    ],
    "mc_no": [
        "mc11a",
        "mc07b",
        "mc02f",
        "mc67x",
        "mc88k",
    ],
    "process": [
        "gd",
        "gd",
        "gd",
        "gd",
        "gd",
    ],
})

# ============================================================
# FILE LOADER
# ============================================================

def load_dataset(file_path: str) -> pd.DataFrame:
    """
    Load a dataset from a supported file format.

    Supported formats:
    - CSV
    - TSV / TXT
    - Parquet
    - Excel
    - JSON
    - Feather
    - Pickle
    """

    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Dataset file not found: {path.resolve()}"
        )

    extension = path.suffix.lower()

    if extension == ".csv":
        return pd.read_csv(path)

    if extension in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")

    if extension in {".parquet", ".pq"}:
        return pd.read_parquet(path)

    if extension in {".xlsx", ".xls", ".xlsm"}:
        return pd.read_excel(path)

    if extension == ".json":
        return pd.read_json(path)

    if extension == ".feather":
        return pd.read_feather(path)

    if extension in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(
        f"Unsupported file format: {extension}\n"
        "Supported formats: CSV, TSV, TXT, Parquet, Excel, "
        "JSON, Feather, Pickle"
    )

# ============================================================
# FRONT-GATE VALIDATION
# ============================================================

def validate_dataset_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Accept only:
    1. The four required columns, or
    2. The four required columns plus the legacy 'registered' column.

    If 'registered' exists, display a warning but do not remove it.
    """

    actual_columns = list(df.columns)

    missing_columns = [
        column for column in REQUIRED_COLUMNS
        if column not in actual_columns
    ]

    unexpected_columns = [
        column for column in actual_columns
        if column not in REQUIRED_COLUMNS + OPTIONAL_COLUMNS
    ]

    has_duplicate_column_names = df.columns.duplicated().any()

    allowed_schema_without_registered = set(REQUIRED_COLUMNS)
    allowed_schema_with_registered = set(REQUIRED_COLUMNS + OPTIONAL_COLUMNS)

    actual_column_set = set(actual_columns)

    schema_is_valid = (
        actual_column_set in [
            allowed_schema_without_registered,
            allowed_schema_with_registered,
        ]
        and not has_duplicate_column_names
        and len(actual_columns) in {4, 5}
    )

    # ========================================================
    # REJECT INVALID DATASET
    # ========================================================

    if not schema_is_valid:
        display(HTML("""
        <div style="
            border: 5px solid #b00020;
            background-color: #ffe6e6;
            padding: 22px;
            margin: 16px 0;
            font-size: 25px;
            font-weight: bold;
            color: #b00020;
            text-align: center;
        ">
            ⚠️ THE DATASET IS NOT IN THE CORRECT PATTERN!!! ⚠️
        </div>
        """))

        print("Required columns:")
        print(REQUIRED_COLUMNS)

        print("\nOptional legacy column:")
        print(OPTIONAL_COLUMNS)

        print("\nYour dataset columns:")
        print(actual_columns)

        if missing_columns:
            print("\nMissing required columns:")
            print(missing_columns)

        if unexpected_columns:
            print("\nUnexpected columns:")
            print(unexpected_columns)

        if has_duplicate_column_names:
            print("\nWarning: duplicate column names were detected.")

        print("\nYour dataset looks like this:")
        display(df.head(5))

        print("\nIt should look similar to this:")
        display(expected_format_example)

        raise ValueError(
            "Dataset rejected: incorrect schema. "
            "Please fix the dataset before running the next step."
        )

    # ========================================================
    # WARN ABOUT LEGACY COLUMN WITHOUT REMOVING IT
    # ========================================================

    if "registered" in df.columns:
        display(HTML("""
        <div style="
            border: 5px solid #f57c00;
            background-color: #fff3e0;
            padding: 22px;
            margin: 16px 0;
            font-size: 22px;
            font-weight: bold;
            color: #e65100;
            text-align: center;
        ">
            ⚠️ WARNING: THE LEGACY "registered" COLUMN WAS DETECTED ⚠️
            <br><br>
            The "registered" column should be removed before proceeding to the next step.
            <br>
            Please use "occurred" as the real machine-event timestamp.
        </div>
        """))

    display(HTML("""
    <div style="
        border: 5px solid #087f23;
        background-color: #e8f5e9;
        padding: 22px;
        margin: 16px 0;
        font-size: 23px;
        font-weight: bold;
        color: #087f23;
        text-align: center;
    ">
        ✅ THE DATASET IS IN CORRECT PATTERN, READY FOR NEXT STEP ✅
    </div>
    """))

    return df

# ============================================================
# RUN FRONT GATE
# ============================================================

df = load_dataset(DATASET_PATH)
df = validate_dataset_schema(df)

# Remove rows with null values in required columns
rows_before_cleaning = len(df)

df = df.dropna(subset=REQUIRED_COLUMNS).copy()

removed_null_rows = rows_before_cleaning - len(df)

# Normalize values
df["occurred"] = pd.to_datetime(
    df["occurred"],
    errors="coerce",
    dayfirst=True,
)

df["mc_status"] = df["mc_status"].astype(str).str.strip()
df["mc_no"] = df["mc_no"].astype(str).str.strip()
df["process"] = df["process"].astype(str).str.strip()

# Remove rows with invalid timestamps
rows_before_timestamp_check = len(df)

df = (
    df.dropna(subset=["occurred"])
      .reset_index(drop=True)
)

removed_invalid_timestamps = (
    rows_before_timestamp_check - len(df)
)

# ============================================================
# SUMMARY
# ============================================================

print(f"Dataset path: {DATASET_PATH}")
print(f"Accepted rows: {len(df):,}")
print(f"Removed rows with null values: {removed_null_rows:,}")
print(
    "Removed rows with invalid occurred timestamps: "
    f"{removed_invalid_timestamps:,}"
)

print("\nFinal dataframe columns:")
print(list(df.columns))

print("\nValidated dataset preview:")
display(df.head(5))